# Mini Project — System Architecture Documentation

## Project: Smart GDD Creator
**Topic:** AI-powered Game Design Document generator — from a one-line game idea to a full GDD

---

## 1. System Overview

The **Smart GDD Creator** is a multi-agent system that transforms a single-sentence game idea into a comprehensive Game Design Document (GDD) — the kind produced by real indie and AAA studios.

The system is orchestrated through four specialized agents: a **Planner** that decomposes the idea into structured design pillars, a **Researcher** that grounds decisions in market data and comparable games, a **Writer** that composes all GDD sections (story, mechanics, UI, monetization), and a **Reviewer** that validates feasibility and coherence.

Agents communicate through a shared state dictionary passed sequentially, with each agent enriching the state before handing off to the next. The final output is a structured, export-ready GDD rendered in a Streamlit UI, where users can edit sections and regenerate any single agent's output on demand.

The system is built with LangChain for agent orchestration, Claude (via Anthropic API) as the underlying LLM, and Streamlit for the interactive front-end.

## 2. Agent Roles

| Agent Name        | Primary Role               | Key Responsibilities                                                                 | Tools / Components Used                          |
|-------------------|----------------------------|--------------------------------------------------------------------------------------|--------------------------------------------------|
| **Planner Agent** | Idea → Design Pillars      | Expand one-line idea into genre, core mechanics loop, target audience, platform, tone | LLM prompt, structured JSON output               |
| **Researcher Agent** | Market & Comp Analysis  | Find comparable games, analyze what made them succeed/fail, identify market gaps     | Web search tool, LLM summarization               |
| **Writer Agent**  | GDD Sections Author        | Write story/lore, character bios, level design, UI flows, monetization strategy      | LLM long-form generation, section templates      |
| **Reviewer Agent**| QA & Feasibility Checker   | Validate scope, flag contradictions, score design pillars, suggest improvements      | LLM critique prompt, rubric scoring              |

## 3. Workflow Diagram

```mermaid
graph TD
    A["💡 User Input\n(One-line game idea)"] --> B["📋 Planner Agent\nGemini Flash — Decomposes idea into design pillars"]
    B --> C["🔍 Researcher Agent\nGemini Flash — Finds comparable games & market data"]
    C --> D["✍️ Writer Agent\nClaude Sonnet — Generates all 8 GDD sections"]
    D --> E["🔎 Reviewer Agent\nClaude Haiku — Scores feasibility 1-10"]
    E --> F{"Score ≥ 7?"}
    F -- Yes --> G["📄 Final GDD\n(Displayed in Streamlit — 5 tabs)"]
    F -- No --> D
    D -. "Max 1 retry" .-> E
    G --> H["📥 Export as JSON"]

    style A fill:#F9A825,color:#212121
    style B fill:#0D9488,color:#fff
    style C fill:#0D9488,color:#fff
    style D fill:#7C3AED,color:#fff
    style E fill:#388E3C,color:#fff
    style F fill:#F59E0B,color:#212121
    style G fill:#22C55E,color:#fff
    style H fill:#1B5E20,color:#fff
```

## 4. Example Execution Trace

**Input:** `"A roguelike dungeon crawler where the dungeon is built from trading cards the player collects"`

---

### Step 1 — Planner Agent Output
```json
{
  "genre": "Roguelike / Deck-builder hybrid",
  "core_loop": "Collect cards → Build dungeon rooms → Survive your own dungeon → Earn new cards",
  "target_audience": "PC gamers 18-35, fans of Slay the Spire and Inscryption",
  "platform": "PC (Steam), potential mobile port",
  "tone": "Dark fantasy with dark humor",
  "pillars": ["Emergent gameplay", "Collection depth", "Narrative surprise"]
}
```

---

### Step 2 — Researcher Agent Output
```
Comparable games: Slay the Spire ($50M+ revenue), Inscryption (IGF Award winner), Monster Train
Market gap identified: No game combines dungeon-building WITH card collection as core mechanic
Risk: High complexity — needs strong onboarding UX
Opportunity: Trading/sharing cards online could drive community virality
```

---

### Step 3 — Writer Agent Output (excerpt)
```
## Story
The player is a cursed archivist trapped in an ever-shifting library-dungeon. 
Every card found is a memory fragment. Survive long enough to reconstruct the truth.

## Core Mechanic
- Draft Phase: Choose 3 cards from a random hand to place as dungeon rooms
- Run Phase: Your hero navigates the dungeon you built, facing your own traps
- Reward Phase: Defeating rooms earns upgraded versions of those cards
```

---

### Step 4 — Reviewer Agent Output
```
Feasibility Score: 8/10
✅ Core loop is clear and differentiated
✅ Target audience is well-defined
⚠️  Monetization section missing — flagged for Writer Agent revision
⚠️  Level progression curve not defined — recommend adding 3-act structure
Action: Sending back to Writer Agent for monetization + progression sections
```

## 5. Coordination Strategy & Memory

**Agent Communication:**  
Agents communicate via a single shared `state` dictionary that is passed sequentially through the pipeline. Each agent reads from the state, appends its outputs under a dedicated key (e.g., `state['planner_output']`), and passes the enriched state to the next agent. This avoids tight coupling — agents don't call each other directly.

**Shared Memory / State Management:**  
The state dictionary acts as the system's working memory. It is initialized with the user's raw input and accumulates structured data at each stage. In the Streamlit UI, this state is stored in `st.session_state` so it persists across user interactions within a session.

**Preventing Infinite Loops:**  
The Reviewer Agent can trigger one revision loop back to the Writer Agent. A `revision_count` counter in the state caps retries at 2. If the GDD still doesn't pass after 2 revisions, the system surfaces the Reviewer's feedback directly to the user for manual intervention rather than looping indefinitely.

**Preventing Duplicated Work:**  
Each agent checks for the presence of its output key in the state before running. If the key exists (e.g., from a cached prior run), the agent skips execution unless the user explicitly clicks "Regenerate" for that section in the UI.

## 6. Challenges & Future Improvements

**Main Challenges:**
- **Prompt chaining consistency:** Keeping the Writer Agent aware of Researcher findings without bloating the context window required careful prompt engineering with summarized state injection.
- **Reviewer subjectivity:** Defining clear, rubric-based criteria for the Reviewer Agent was harder than expected — "good game design" is inherently subjective, so scoring had to be broken into objective sub-dimensions (scope, completeness, contradiction-check).
- **Streamlit state management:** Preserving agent outputs across UI re-renders without re-running the full pipeline required careful use of `st.session_state` and conditional execution logic.

**Future Improvements (Production-Grade):**
- Add a **parallel Research Agent** using async calls to query multiple game databases simultaneously (IGDB, Steam Spy, Metacritic)
- Implement **long-term memory** (e.g., via a vector database) so the system learns from previously generated GDDs and improves recommendations over time
- Add a **Visual Concept Agent** that generates concept art prompts (or calls an image generation API) for key characters and environments
- Build a **PDF/Notion export** pipeline so the GDD is immediately shareable in a professional format
- Add **user authentication** so multiple team members can collaborate on the same GDD in real time

## Checklist

- [x] System Overview written
- [x] Agent Roles table completed
- [x] Workflow diagram included
- [x] Example execution trace provided
- [x] Coordination & Memory section completed
- [x] Challenges & Improvements section completed